# Spread Locator: Statistical Distribution Analysis
**Objective:** Analyze probability distributions, Q-Q plots, statistical transformations, and probability functions on transaction data.

In [ ]:
import pandas as pd
import numpy as np
import scipy.stats as stats
import statsmodels.api as sm
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
%matplotlib inline

# Load dataset
df = pd.read_csv('../data/spread_locator_dataset.csv')
df.head()

## 1. Summary Statistics & Data Inspection

In [ ]:
print(df.info())
print("\nSummary Statistics:")
display(df['transaction_amount'].describe())
print(f"Skewness: {df['transaction_amount'].skew():.4f}")
print(f"Kurtosis: {df['transaction_amount'].kurtosis():.4f}")

## 2. Fit Bernoulli & Binomial Distributions

In [ ]:
# Bernoulli fit on transaction_status
p_success = (df['transaction_status'] == 'Success').mean()
print(f"Bernoulli parameter p (Success Rate): {p_success:.4f}")

# Binomial fit on transaction_count
n_trials = df['transaction_count'].max()
p_bin = df['transaction_count'].mean() / n_trials
print(f"Binomial estimated p: {p_bin:.4f} for n={n_trials}")

plt.figure(figsize=(7, 4))
sns.countplot(x='transaction_status', data=df, palette='Blues_r')
plt.title(f"Bernoulli Distribution: Success ({p_success*100:.1f}%) vs Fail ({(1-p_success)*100:.1f}%)")
plt.show()

## 3. Fit Poisson Distribution (Daily Arrivals)

In [ ]:
daily_counts = df.groupby('transaction_date').size()
lambda_daily = daily_counts.mean()
print(f"Poisson lambda (daily transactions): {lambda_daily:.4f}")
print(f"Daily count variance: {daily_counts.var():.4f}")

plt.figure(figsize=(8, 4))
sns.histplot(daily_counts, discrete=True, kde=False, color='steelblue')
plt.axvline(lambda_daily, color='red', linestyle='--', label=f'Mean Lambda ({lambda_daily:.2f})')
plt.title('Daily Transactions Count (Poisson Distribution)')
plt.xlabel('Transactions per Day')
plt.legend()
plt.show()

## 4. Q-Q Plot & Normality Testing

In [ ]:
shapiro_w, shapiro_p = stats.shapiro(df['transaction_amount'])
print(f"Raw Amount Shapiro-Wilk Test: W={shapiro_w:.4f}, p-value={shapiro_p:.4e}")

fig, ax = plt.subplots(figsize=(6, 5))
sm.qqplot(df['transaction_amount'], line='s', ax=ax)
plt.title('Q-Q Plot for Raw Transaction Amount')
plt.show()

## 5. Box-Cox Transformation to Stabilize Variance

In [ ]:
transformed_amt, best_lambda = stats.boxcox(df['transaction_amount'])
print(f"Optimal Box-Cox lambda: {best_lambda:.4f}")
print(f"Transformed Skewness: {pd.Series(transformed_amt).skew():.4f}")
w_bc, p_bc = stats.shapiro(transformed_amt)
print(f"Transformed Shapiro-Wilk: W={w_bc:.4f}, p-value={p_bc:.4f}")

fig, ax = plt.subplots(figsize=(6, 5))
sm.qqplot(transformed_amt, line='s', ax=ax)
plt.title(f'Q-Q Plot after Box-Cox Transformation (λ = {best_lambda:.4f})')
plt.show()

## 6. Distribution Modeling: Log-Normal vs Power Law (Pareto)

In [ ]:
shape_ln, loc_ln, scale_ln = stats.lognorm.fit(df['transaction_amount'], floc=0)
ks_ln = stats.kstest(df['transaction_amount'], 'lognorm', args=(shape_ln, loc_ln, scale_ln))
ks_norm = stats.kstest(df['transaction_amount'], 'norm', args=(df['transaction_amount'].mean(), df['transaction_amount'].std()))

print(f"Log-Normal Fit KS Test: Stat={ks_ln.statistic:.4f}, p-value={ks_ln.pvalue:.4f}")
print(f"Normal Fit KS Test: Stat={ks_norm.statistic:.4f}, p-value={ks_norm.pvalue:.4e}")

## 7. Z-Score Probability Calculation (Exceeding ₹5,000)

In [ ]:
mean_val = df['transaction_amount'].mean()
std_val = df['transaction_amount'].std()
z_5000 = (5000 - mean_val) / std_val
p_norm_5000 = 1 - stats.norm.cdf(z_5000)
p_emp_5000 = (df['transaction_amount'] > 5000).mean()
p_lognorm_5000 = 1 - stats.lognorm.cdf(5000, shape_ln, loc=loc_ln, scale=scale_ln)

print(f"Z-score for ₹5,000: {z_5000:.4f}")
print(f"Normal Distribution P(X > 5000): {p_norm_5000*100:.2f}%")
print(f"Empirical Probability P(X > 5000): {p_emp_5000*100:.2f}%")
print(f"Log-Normal Model P(X > 5000): {p_lognorm_5000*100:.2f}%")

## 8. PDF & CDF Visualization and Interpretation

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
x_vals = np.linspace(df['transaction_amount'].min(), df['transaction_amount'].max(), 500)

# PDF
sns.histplot(df['transaction_amount'], stat='density', kde=True, ax=axes[0], color='skyblue', label='Empirical Density')
axes[0].plot(x_vals, stats.lognorm.pdf(x_vals, shape_ln, loc=loc_ln, scale=scale_ln), 'r--', lw=2, label='Fitted Log-Normal')
axes[0].set_title('Probability Density Function (PDF)')
axes[0].set_xlabel('Transaction Amount (₹)')
axes[0].legend()

# CDF
sns.ecdfplot(df['transaction_amount'], ax=axes[1], color='navy', label='Empirical CDF')
axes[1].plot(x_vals, stats.lognorm.cdf(x_vals, shape_ln, loc=loc_ln, scale=scale_ln), 'r--', lw=2, label='Log-Normal CDF')
axes[1].axvline(5000, color='green', linestyle=':', label='₹5,000 Threshold')
axes[1].set_title('Cumulative Distribution Function (CDF)')
axes[1].set_xlabel('Transaction Amount (₹)')
axes[1].legend()

plt.tight_layout()
plt.show()